# 04 — Duplicates and groups
Audit DeepPCB và PKU riêng biệt, kể cả near-duplicate (pHash Hamming <= 5). Group theo tiền tố số đầu tiên trong filename (`01_missing_hole_01.jpg` → `01`); nếu không có tiền tố, dùng component liên thông của near-duplicate làm group phụ. Group prefix là giả thuyết template board, không chứng minh cùng PCB vật lý. Cluster pHash là quan hệ bắc cầu, không đảm bảo mọi cặp trong cluster có distance <= 5. Không so sánh hai dataset với nhau.

In [1]:
from pathlib import Path
from collections import Counter
import re
import pandas as pd
from PIL import Image
from IPython.display import display, Markdown
try:
    import imagehash
except ImportError as exc:
    raise ImportError('Cài imagehash bằng `python -m pip install ImageHash` rồi chạy lại notebook 04.') from exc
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/raw/DeepPCB').is_dir()), None)
if ROOT is None: raise FileNotFoundError('Run from inside the project')
DATASETS = {'PKU': 'PKU-Market-PCB(Data enhanced version)', 'DeepPCB': 'DeepPCB'}
SPLITS = ('train', 'valid', 'test')
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
def near_pairs(hashes, threshold=5):
    # BK tree: exact Hamming-radius search over all hashes (including identical hashes).
    root, pairs = None, []
    for idx, value in enumerate(hashes):
        if root is None:
            root = [value, [idx], {}]
            continue
        node = root
        while True:
            distance = (value ^ node[0]).bit_count()
            if distance <= threshold:
                pairs.extend((previous, idx, distance) for previous in node[1])
            if distance == 0:
                node[1].append(idx)
                break
            child = node[2].get(distance)
            if child is None:
                node[2][distance] = [value, [idx], {}]
                break
            node = child
        # Search all other branches within the query radius.
        stack = [root]
        while stack:
            candidate = stack.pop()
            d = (value ^ candidate[0]).bit_count()
            if d <= threshold and candidate is not node and idx not in candidate[1]:
                pairs.extend((previous, idx, d) for previous in candidate[1])
            stack.extend(child for edge, child in candidate[2].items() if d - threshold <= edge <= d + threshold)
    return list(dict.fromkeys(pairs))


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
all_groups = []
for dataset_name, folder in DATASETS.items():
    files = [(split, p) for split in SPLITS for p in sorted((ROOT / 'data/raw' / folder / split / 'images').iterdir()) if p.is_file() and p.suffix.lower() in EXTS]
    hashes = []
    for split, path in files:
        with Image.open(path) as image:
            hashes.append(imagehash.phash(image))
    pairs = near_pairs([int(str(h), 16) for h in hashes])
    parent = list(range(len(files)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for i, j, distance in pairs:
        a, b = find(i), find(j)
        if a != b: parent[b] = a
    groups = []
    for i, ((split, path), h) in enumerate(zip(files, hashes)):
        prefix = re.match(r'^(\d+)(?:_|$)', path.stem)
        group_id = f'prefix_{prefix.group(1)}' if prefix else f'phash_cluster_{min(j for j in range(len(files)) if find(j) == find(i)):05d}'
        groups.append({'image_path': str(path.relative_to(ROOT)), 'split': split, 'dataset_name': dataset_name, 'group_id': group_id, 'phash': str(h), 'derived_by': 'filename_prefix' if prefix else 'phash_cluster'})
    frame = pd.DataFrame(groups)
    all_groups.append(frame)
    display(Markdown(f'## {dataset_name}: {len(frame)} ảnh; {frame.group_id.nunique()} group; {len(pairs)} near-duplicate pairs (distance <= 5)'))
    display(frame.groupby('group_id').size().rename('images').reset_index())
    crossing = [(i, j, d) for i, j, d in pairs if files[i][0] != files[j][0]]
    display(Markdown(f'Near-duplicate khác split: **{len(crossing)} cặp**; cùng split: **{len(pairs) - len(crossing)} cặp**. Hiển thị 30 cặp cross-split đầu tiên (đầy đủ có thể tái tạo bằng `pairs`).'))
    display(pd.DataFrame([{'image_a': str(files[i][1].relative_to(ROOT)), 'image_b': str(files[j][1].relative_to(ROOT)), 'distance': d} for i, j, d in crossing[:30]], columns=['image_a', 'image_b', 'distance']))
groups_path = ROOT / 'data/splits/groups.csv'
pd.concat(all_groups, ignore_index=True).to_csv(groups_path, index=False)
display(Markdown(f'Đã xuất `{groups_path.relative_to(ROOT)}`.'))
audit_doc = ROOT / 'docs/data_audit.md'
if audit_doc.exists():
    paragraph = 'Group được suy ra bằng tiền tố số đầu tiên của filename (nếu thiếu thì dùng component pHash); kiểm chứng bằng imagehash.phash và near-duplicate Hamming <= 5 trong từng dataset, kể cả giữa các split; giới hạn của cách suy ra này là tiền tố chỉ là giả thuyết template và pHash có thể nhầm ảnh tương tự, cluster bắc cầu không đảm bảo cùng một board.'
    existing = audit_doc.read_text(encoding='utf-8')
    if paragraph not in existing:
        with audit_doc.open('a', encoding='utf-8') as out: out.write(('\n' if existing and not existing.endswith('\n') else '') + '\n' + paragraph + '\n')
else:
    display(Markdown('Chưa ghi docs/data_audit.md: file chưa tồn tại; cần được người dùng cho phép trước khi tạo.'))


## PKU: 3504 ảnh; 10 group; 82495 near-duplicate pairs (distance <= 5)

,group_id,images
0,prefix_01,720
1,prefix_04,720
2,prefix_05,360
3,prefix_06,366
4,prefix_07,250
5,prefix_08,240
6,prefix_09,240
7,prefix_10,128
8,prefix_11,240
9,prefix_12,240


Near-duplicate khác split: **20121 cặp**; cùng split: **62374 cặp**. Hiển thị 30 cặp cross-split đầu tiên (đầy đủ có thể tái tạo bằng `pairs`).

,image_a,image_b,distance
0,data/raw/PKU-Market-PCB(Data enhanced version)...,data/raw/PKU-Market-PCB(Data enhanced version)...,4
1,data/raw/PKU-Market-PCB(Data enhanced version)...,data/raw/PKU-Market-PCB(Data enhanced version)...,4
2,data/raw/PKU-Market-PCB(Data enhanced version)...,data/raw/PKU-Market-PCB(Data enhanced version)...,4
3,data/raw/PKU-Market-PCB(Data enhanced version)...,data/raw/PKU-Market-PCB(Data enhanced version)...,4
4,data/raw/PKU-Market-PCB(Data enhanced version)...,data/raw/PKU-Market-PCB(Data enhanced version)...,4
5,data/raw/PKU-Market-PCB(Data enhanced version)...,data/raw/PKU-Market-PCB(Data enhanced version)...,4
6,data/raw/PKU-Market-PCB(Data enhanced version)...,data/raw/PKU-Market-PCB(Data enhanced version)...,4
7,data/raw/PKU-Market-PCB(Data enhanced version)...,data/raw/PKU-Market-PCB(Data enhanced version)...,4
8,data/raw/PKU-Market-PCB(Data enhanced version)...,data/raw/PKU-Market-PCB(Data enhanced version)...,4
9,data/raw/PKU-Market-PCB(Data enhanced version)...,data/raw/PKU-Market-PCB(Data enhanced version)...,4


## DeepPCB: 1500 ảnh; 1 group; 16 near-duplicate pairs (distance <= 5)

,group_id,images
0,prefix_01,1500


Near-duplicate khác split: **6 cặp**; cùng split: **10 cặp**. Hiển thị 30 cặp cross-split đầu tiên (đầy đủ có thể tái tạo bằng `pairs`).

,image_a,image_b,distance
0,data/raw/DeepPCB/train/images/01_PCB__1124.jpg,data/raw/DeepPCB/valid/images/01_PCB__1156.jpg,4
1,data/raw/DeepPCB/train/images/01_PCB__1155.jpg,data/raw/DeepPCB/test/images/01_PCB__1187.jpg,4
2,data/raw/DeepPCB/train/images/01_PCB__1491.jpg,data/raw/DeepPCB/test/images/01_PCB__1395.jpg,4
3,data/raw/DeepPCB/train/images/01_PCB__1394.jpg,data/raw/DeepPCB/test/images/01_PCB__1490.jpg,2
4,data/raw/DeepPCB/train/images/01_PCB__1396.jpg,data/raw/DeepPCB/test/images/01_PCB__1492.jpg,4
5,data/raw/DeepPCB/train/images/01_PCB__306.jpg,data/raw/DeepPCB/test/images/01_PCB__358.jpg,4


Đã xuất `data/splits/groups.csv`.

Chưa ghi docs/data_audit.md: file chưa tồn tại; cần được người dùng cho phép trước khi tạo.